# EICC GRPO Training Notebook

Colab-ready notebook for the Enterprise Incident Command Center:
1. Clone repo & install dependencies
2. Verify environment works (dry-run)
3. Run baseline evaluation
4. Train with GRPO
5. Run post-training evaluation
6. Plot reward curves and inspect artifacts

In [ ]:
# Step 1: Clone your repo
!git clone https://github.com/anujkamaljain/OpenEnv-Customer-Support.git
%cd OpenEnv-Customer-Support

In [ ]:
# Step 2: Install training dependencies (clean path)
# If Colab asks for runtime restart, restart and rerun from Step 1.
%pip install -q -U pip setuptools wheel jedi
%pip install -q -U "pydantic>=2.12,<3" "click<8.2"
%pip install -q -U "trl==0.15.2" "transformers==4.51.3" datasets peft bitsandbytes accelerate matplotlib llm-blender mergekit
%pip install -q -U unsloth

In [ ]:
# Step 3: Verify training stack + environment (quick sanity check)
import importlib.metadata as im
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

print("unsloth:", im.version("unsloth"))
print("trl:", im.version("trl"))
print("transformers:", im.version("transformers"))

!python train.py --iterations 1 --episodes 1 --k 2 --dry-run

In [ ]:
# Step 4: Run baseline evaluation (before training)
!python evaluate.py --policy baseline --episodes-per-difficulty 5 --output-dir artifacts/eval

In [ ]:
# Step 5: GRPO Training (~6-8 hours on T4, ~2-3 hours on V100)
# Reduce iterations/episodes if short on time:
#   --iterations 10 --episodes 15 --k 2 --seed 7
!python train.py --iterations 20 --episodes 30 --k 4 --eval-episodes 5 --seed 7 --max-completion-length 128 --output-dir artifacts/train

In [ ]:
# Step 6: Run post-training comparison evaluation
# Recommended: compare baseline vs trained checkpoint adapter
!python evaluate.py --policy compare --compare-trained-policy trained_checkpoint --checkpoint-dir artifacts/train/trained_adapter --checkpoint-base-model Qwen/Qwen2.5-3B-Instruct --episodes-per-difficulty 5 --plot --output-dir artifacts/eval

In [ ]:
# Step 7: View results
from pathlib import Path
import json

baseline_path = Path("artifacts/eval/baseline_report.json")
trained_path = Path("artifacts/eval/trained_report.json")
if baseline_path.exists() and trained_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    trained = json.loads(trained_path.read_text(encoding="utf-8"))
    print("Baseline avg_normalized_reward:", baseline["avg_normalized_reward"])
    print("Trained avg_normalized_reward:", trained["avg_normalized_reward"])
    print("Behavior examples:")
    for line in trained.get("behavior_examples", []):
        print("-", line)
else:
    print("Run evaluation cells first.")

In [ ]:
# Step 8: Display reward curve
from pathlib import Path
from IPython.display import Image, display

curve = Path("artifacts/eval/reward_curves.png")
if curve.exists():
    display(Image(filename=str(curve)))
else:
    print("Run compare evaluation with --plot first.")

In [ ]:
# Step 9 (optional): Two-seed reproducibility run
# Use this for final submission metrics (mean +/- std)
!python train.py --iterations 10 --episodes 15 --k 2 --seed 7 --max-completion-length 128 --output-dir artifacts/train_seed7
!python train.py --iterations 10 --episodes 15 --k 2 --seed 17 --max-completion-length 128 --output-dir artifacts/train_seed17

In [ ]:
# Step 10 (optional): Summarize two-seed metrics
from pathlib import Path
import json, math

paths = [
    Path("artifacts/train_seed7/trained_report.json"),
    Path("artifacts/train_seed17/trained_report.json"),
]
vals_norm, vals_raw = [], []
for p in paths:
    if p.exists():
        obj = json.loads(p.read_text(encoding="utf-8"))
        vals_norm.append(float(obj["avg_normalized_reward"]))
        vals_raw.append(float(obj.get("avg_raw_reward", 0.0)))

if len(vals_norm) == 2:
    mean_norm = sum(vals_norm) / 2
    mean_raw = sum(vals_raw) / 2
    std_norm = math.sqrt(sum((x - mean_norm) ** 2 for x in vals_norm) / 2)
    std_raw = math.sqrt(sum((x - mean_raw) ** 2 for x in vals_raw) / 2)
    print(f"avg_normalized_reward: {mean_norm:.4f} +/- {std_norm:.4f}")
    print(f"avg_raw_reward: {mean_raw:.4f} +/- {std_raw:.4f}")
else:
    print("Run Step 9 first to produce both seed reports.")